<a href="https://colab.research.google.com/github/Squad-Nina-da-Hora/wmc-desafio-previsao-demencia/blob/main/analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# **Análise de Previsão de Demência**
---


🎯 **Objetivo:**  Prever sinais de demência, através de informações clínicas e demográficas de pacientes com potencial risco de Alzheimer (OASIS).


---


Desafio Estatística com Python - Classificação

Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## 1. Configurações Iniciais

Variáveis da base de dados:

- `Subject ID`: Identificador único do paciente
- `MRI ID`: Identificador único do exame
- `Group` (alvo): Classificação do paciente
  - `Nondemented` - será tratada para variável binária 0
  - `Demented` e `Converted` - serão tratadas para variável binária 1
- `Visit`: Identificador da visita de cada paciente
- `MR Delay`: Intervalo em dias entre os exames
- `M/F`: Gênero (M: masculino, F: feminino)
- `Hand`: Mão dominante
- `Age`: Idade do paciente (numérico)
- `EDUC`: Anos de escolaridade (numérico)
- `SES`: Status socioeconômico (1 a 5)
- `MMSE`: Escore do Mini Exame do Estado Mental (0 a 30)
- `CDR`: Clinical Dementia Rating (0 a 3)
- `eTIV`: Volume intracraniano estimado
- `nWBV`: Proporção de volume cerebral normalizado
- `ASF`: Fator de escala anatômica

In [ ]:
# ==============================
# IMPORTACOES
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from IPython.display import display, Markdown

# Carregamento da base de dados
arquivo = 'oasis_longitudinal'
url = f'https://raw.githubusercontent.com/Squad-Nina-da-Hora/wmc-desafio-previsao-demencia/main/{arquivo}.csv'
df = pd.read_csv(url)

In [ ]:
# ==============================
# VARIAVEIS PARA REUTILIZACAO
# ==============================

# data: dataframe contendo apenas as colunas de interesse
# nulos: colunas que possuem valores nulos
# var_features: seleção do df sem a variável alvo
# corr_rank: ranking de correlação com group
var_alvo = 'Group'

# Configurações visuais dos gráficos

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
paleta = 'flare'
cores = sns.color_palette(paleta, n_colors=2)

##  2. Análise exploratória dos dados

In [ ]:
# Conhecendo os dados

n_cols = df.shape[1]
print(f'\nTotal de linhas: {df.shape[0]}')
print(f'Total de colunas: {n_cols}')
print('-' * 50)

# Verificando duplicatas

duplicados = df.duplicated().sum()
print(f'\nLinhas duplicadas na base: {duplicados}')

In [ ]:
# Verificando tipagem e nulos

info_df = pd.DataFrame({
    'Tipo': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df)) * 100,
    'Valores Únicos': df.nunique()
})
print('\n--- Diagnóstico de Tipagem e Qualidade ---\n')
display(info_df)

nulos = df.columns[df.isna().any()].tolist()
# Armazenando colunas com valores nulos
print(f'Colunas com nulos identificadas: {nulos}')

# Grafico de nulos
msno.matrix(df, figsize=(10, 5), fontsize=9,
            color=sns.color_palette(paleta, n_colors=n_cols)[int(n_cols / 2)])
plt.title('Visualização de Valores Ausentes', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Conhecendo os dados

df.head()

In [ ]:
# Conhecendo os dados

df.describe()

In [ ]:
# Conhecendo os dados

df.describe(include=['object', 'string'])

In [ ]:
# Conhecendo os dados

df['Group'].unique()

In [ ]:
# Conhecendo os dados

df['M/F'].unique()

In [ ]:
# Conhecendo os dados

df['Hand'].unique()

In [ ]:
# Selecionando apenas colunas de interesse

data = df.drop(['Subject ID', 'MRI ID', 'Visit', 'MR Delay', 'Hand'], axis=1)
data.head()

In [ ]:
# Variável categórica: % de group

data['Group'].value_counts(normalize=True)

In [ ]:
# Variável categórica: tratamento de group

data['Group'] = data['Group'].map({'Demented': 1, 'Converted': 1, 'Nondemented': 0})
data.head()

In [ ]:
# Variável categórica: % de gênero

data['M/F'].value_counts(normalize=True)

In [ ]:
# Variável categórica: tratamento de gênero

data['M/F'] = data['M/F'].map({'M': 1, 'F': 0})
data.head()

In [ ]:
# Fazendo a seleção do df sem a variável alvo (para reutilização)

var_features = [col for col in data.columns if col != var_alvo]
var_features

In [ ]:
# Calculando a matriz de correlação

corr = data.corr()
corr

In [ ]:
# Exibindo heatmap

plt.figure(figsize=(10,8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt='.2f', linewidth=0.5)
plt.title('Matriz de correlação', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
# Analisando distribuição e desbalanceamento de Group

plt.figure(figsize=(8, 6))
ax = sns.countplot(data=data, x=var_alvo, palette=cores, hue=var_alvo, legend=False)
plt.title(f'Distribuição da Variável Alvo ({var_alvo})', fontsize=12, fontweight='bold')
plt.xlabel(f'{var_alvo}')
plt.ylabel('Contagem')

# Adicionando porcentagens nas barras
total = len(data)
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{(height/total)*100:.1f}%',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom', xytext=(0, 3),
                textcoords='offset points')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
axes = axes.flatten()

for i, var in enumerate(var_features):
    sns.boxplot(data=data, x=var_alvo, y=var, ax=axes[i], palette=cores, hue=var_alvo, legend=False)
    axes[i].set_title(f'Boxplot: {var} por {var_alvo}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Histogramas

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
axes = axes.flatten()

for i, var in enumerate(var_features):
    sns.histplot(data=data, x=var, hue=var_alvo, kde=True, ax=axes[i], palette=cores, element='step')
    axes[i].set_title(f'Distribuição: {var}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Exibindo dispersões

sns.pairplot(data, hue=var_alvo, diag_kind='hist', palette=cores)

In [ ]:
# Ranqueando os principais preditores

corr_rank = corr['Group'].abs().drop('Group').sort_values(ascending=False)
corr_rank

In [ ]:
display(Markdown(
        f'Com base na análise exploratória e no ranqueamento de correlação linear, os principais candidatos a preditores da variável alvo {var_alvo} são:\n'
        f'- {corr_rank.index[0]} (|r| = {corr_rank.iloc[0]:.2f}), \n'
        f'- {corr_rank.index[1]} (|r| = {corr_rank.iloc[1]:.2f}) e \n'
        f'- {corr_rank.index[2]} (|r| = {corr_rank.iloc[2]:.2f}).\n\n'
        f'Adicionalmente, as variáveis {corr_rank.index[3]} (|r| = {corr_rank.iloc[3]:.2f}) e {corr_rank.index[4]} (|r| = {corr_rank.iloc[4]:.2f}) demonstram associação intermediária, enquanto as demais características ({', '.join(corr_rank.index[5:])}) apresentam fraca correlação linear direta (|r| < 0.10) com o diagnóstico do grupo.'
))

## 3. Modelo de ML

Optamos por separar 75% dos dados para treino e 25% para teste.

In [ ]:
def tratar_nulos(dados_treino, dados_teste=None, tendencia='media'):
  """
  Trata valores ausentes usando a media, mediana ou moda, calculada apenas no
  treino e aplicada em treino e teste (evita vazamento de dados).

  Parametros:
    dados_treino (pd.DataFrame): conjunto de treino.
    dados_teste (pd.DataFrame, opcional): conjunto de teste.
    tendencia (str, opcional): 'media', 'mediana' ou 'moda'. Padrao: 'media'.

  Retorno: tupla (X_train, X_test) com nulos preenchidos.
  """
  if tendencia == 'moda':
    valores = dados_treino.mode().iloc[0]
  else:
    valores = getattr(dados_treino, tendencia)()

  dados_treino = dados_treino.fillna(valores)

  if dados_teste is not None:
    dados_teste = dados_teste.fillna(valores)

  return dados_treino, dados_teste


def preprocessar_dados(dados, alvo, test_size=0.2, random_state=42):
  """
  Preprocessa o dataframe para modelagem:
  1. Separa variaveis preditoras (X) e alvo (y).
  2. Faz a separacao estratificada dos dados em treino e teste.
  3. Normaliza as variaveis usando StandardScaler (ajustado no treino).
  4. Reconstroi dfs normalizados com colunas e indices originais.

  Parametros:
    dados (pd.DataFrame): df de entrada, ja tratado.
    alvo (str): nome da coluna davariavel alvo.
    test_size (float, opcional): proporcao dos dados para o conjunto de teste. Padrao: 0.2.

  Retorno: tupla: (X_train_scaled, X_test_scaled, y_train, y_test)
  """

  # Separa as variaveis para treino e teste
  X = dados.drop(columns=[alvo])
  y = dados[alvo]

  X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
  )

  # Trata nulos: moda calculada SO no treino, aplicada nos dois
  X_train, X_test = tratar_nulos(X_train, X_test, 'moda')

  # Normaliza com StandardScaler pra nao vazar dado
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)    # fit + transform no treino
  X_test_scaled = scaler.transform(X_test)          # so transform no teste

  # Traz o df de volta com os nomes das colunas e indices originais
  X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
  X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

  return X_train_scaled, X_test_scaled, y_train, y_test

In [ ]:
X_train_scaled, X_test_scaled, y_train, y_test = preprocessar_dados(data.drop(['CDR'], axis=1), var_alvo, 0.25)

print(f'Tamanho do conjunto de treino: {X_train_scaled.shape[0]} amostras | Nulos: {X_train_scaled.isnull().sum().sum()}')
print(f'Tamanho do conjunto de teste: {X_test_scaled.shape[0]} amostras | Nulos: {X_test_scaled.isnull().sum().sum()}')

## Modelagem Preditiva - Pergunta 2

Nesta etapa, instanciamos os três algoritmos solicitados (Regressão Logística, Árvore de Decisão e Random Forest).

In [ ]:
# Instanciando e treinando
log_reg = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# Predições
y_pred_log = log_reg.predict(X_test_scaled)
y_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1]

# Métricas
acc_log = accuracy_score(y_test, y_pred_log)
prec_log = precision_score(y_test, y_pred_log, pos_label=1)
rec_log = recall_score(y_test, y_pred_log, pos_label=1)
f1_log = f1_score(y_test, y_pred_log, pos_label=1)
auc_log = roc_auc_score(y_test, y_prob_log)

print("Regressão Logística treinada com sucesso!")

Treinamento e Cálculo de Métricas

In [ ]:
# Instanciando e treinando
dt_tree = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dt_tree.fit(X_train_scaled, y_train)

# Predições
y_pred_dt = dt_tree.predict(X_test_scaled)
y_prob_dt = dt_tree.predict_proba(X_test_scaled)[:, 1]

# Métricas
acc_dt = accuracy_score(y_test, y_pred_dt)
prec_dt = precision_score(y_test, y_pred_dt, pos_label=1)
rec_dt = recall_score(y_test, y_pred_dt, pos_label=1)
f1_dt = f1_score(y_test, y_pred_dt, pos_label=1)
auc_dt = roc_auc_score(y_test, y_prob_dt)

print("Árvore de Decisão treinada com sucesso!")

In [ ]:
# Instanciando e treinando
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42)
rf_clf.fit(X_train_scaled, y_train)

# Predições
y_pred_rf = rf_clf.predict(X_test_scaled)
y_prob_rf = rf_clf.predict_proba(X_test_scaled)[:, 1]

# Métricas
acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf, pos_label=1)
rec_rf = recall_score(y_test, y_pred_rf, pos_label=1)
f1_rf = f1_score(y_test, y_pred_rf, pos_label=1)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Random Forest treinado com sucesso!")

Exibição da Tabela Comparativa

In [ ]:
dados_comparativos = {
    "Modelo": ["Regressão Logística", "Árvore de Decisão", "Random Forest"],
    "Acurácia": [acc_log, acc_dt, acc_rf],
    "Precisão": [prec_log, prec_dt, prec_rf],
    "Recall (Sensibilidade)": [rec_log, rec_dt, rec_rf],
    "F1-Score": [f1_log, f1_dt, f1_rf],
    "AUC-ROC": [auc_log, auc_dt, auc_rf]
}

df_comparativo = pd.DataFrame(dados_comparativos).set_index("Modelo")

tabela_estilizada = (
    df_comparativo.style
    # Define fundo branco e texto escuro para TODAS as células do corpo
    .set_properties(**{
        'background-color': '#ffffff',
        'color': '#212529',
        'padding': '8px',
        'text-align': 'center'
    })
    # Aplica a cor de destaque (rosa/coral) por cima nos maiores valores de cada coluna
    .highlight_max(axis=0, color='#fde2e4')
    .format({
        "Acurácia": "{:.1%}",
        "Precisão": "{:.1%}",
        "Recall (Sensibilidade)": "{:.1%}",
        "F1-Score": "{:.1%}",
        "AUC-ROC": "{:.3f}"
    })
    .set_caption("<b>Tabela Comparativa de Desempenho dos Modelos de ML</b>")
    .set_table_styles([
        # Título
        {'selector': 'caption', 'props': [
            ('font-size', '14px'),
            ('text-align', 'center'),
            ('margin-bottom', '10px'),
            ('color', '#ffffff'), # Texto do título em branco para se destacar no tema escuro
            ('font-weight', 'bold')
        ]},
        # Cabeçalho da tabela e coluna de índices
        {'selector': 'th', 'props': [
            ('background-color', '#f1f3f5'),
            ('color', '#212529'),
            ('text-align', 'center'),
            ('font-weight', 'bold'),
            ('padding', '8px')
        ]},
        # Borda da tabela
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('border', '1px solid #dee2e6')
        ]}
    ])
)

display(tabela_estilizada)

A tabela apresenta o comparativo de métricas dos três modelos testados (**Regressão Logística**, **Árvore de Decisão** e **Random Forest**). Como o objetivo é prever sinais de demência, a avaliação foca especialmente na **Classe 1 (Demented/Converted)** — pacientes identificados com declínio cognitivo — mantendo o equilíbrio em relação à **Classe 0 (Nondemented)**.

### Interpretação das Métricas Chave:

* **Recall (Sensibilidade) da Classe 1:** Métrica crítica no contexto da saúde, pois indica a capacidade de identificar corretamente os pacientes doentes e evitar falsos negativos.
  * A **Regressão Logística** obteve o maior Recall (**71,7%**), seguida pelo **Random Forest** (**67,4%**) e pela **Árvore de Decisão** (**58,7%**).

* **Precisão (Precision) da Classe 1:** Mede a confiabilidade de um diagnóstico positivo (evitar alarmes falsos em pacientes saudáveis).
  * O **Random Forest** destacou-se com a maior precisão (**86,1%**), seguido pela **Regressão Logística** (**78,6%**) e pela **Árvore de Decisão** (**77,1%**).

* **Acurácia Geral e AUC-ROC:**
  * O **Random Forest** alcançou a melhor **Acurácia Geral** (**78,7%**) e o maior **F1-Score** (**75,6%**).
  * A **Regressão Logística** obteve a melhor capacidade de separação global (**AUC-ROC de 0.883**).

---

## Avaliação dos Modelos

1. Regressão Logística

In [ ]:
# Métricas

print(classification_report(y_test, y_pred_log))
print('\n' + '-'*60 + '\n')

# Matriz de Confusão

matriz_conf_log = confusion_matrix(y_test, y_pred_log)
plt.figure()
sns.heatmap(matriz_conf_log, annot=True, fmt='d', cmap=paleta, cbar=False, annot_kws={"size": 14})
plt.title('Matriz de Confusão - Regressão Logística', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n' + '-'*60 + '\n')

# Peso das variáveis

peso_log = pd.DataFrame({
    'Variável': X_train_scaled.columns,
    'Peso': log_reg.coef_[0]
}).sort_values(by='Peso', ascending=False)

plt.figure()
ax = sns.barplot(data=peso_log, x='Peso', y='Variável', hue='Variável', palette=paleta)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', label_type='edge', fontsize=10, weight='bold', color='black', padding=2)
plt.title('Peso das Variáveis - Regressão Logística', fontsize=14, fontweight='bold')
plt.xlabel('Peso', fontsize=12, fontweight='bold')
plt.ylabel('Variável', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
"""
Avaliação da Regressão Logística - Diagnóstico
=============================================================
  1. Odds Ratios -> facilita a interpretação dos coeficientes
  2. Modelo via statsmodels -> traz p-valores e intervalos de confiança

"""

# =====================================================================
# Modelo via statsmodels -> coeficientes, p-valores e IC 95%
# =====================================================================
# Para fins de diagnóstico/interpretação estatística.

X_train_const = sm.add_constant(X_train_scaled)  # adiciona o intercepto
modelo_sm = sm.Logit(y_train, X_train_const).fit(disp=False)

print("=== Resumo estatístico do modelo (statsmodels) ===")
print(modelo_sm.summary())
print()


# =====================================================================
# 2. Odds Ratios + Intervalo de Confiança (95%)
# =====================================================================
# Odds Ratio (OR) = exp(coeficiente)
#   OR > 1 -> aumenta a chance do evento (Demented = 1)
#   OR < 1 -> diminui a chance do evento
#   OR = 1 -> sem efeito
#
# Como as variáveis numéricas foram padronizadas, o OR representa o
# efeito de "1 desvio-padrão de aumento" na variável.

params = modelo_sm.params
conf = modelo_sm.conf_int()
conf.columns = ["IC 2.5%", "IC 97.5%"]

odds_ratios = pd.DataFrame({
    "Coeficiente (β)": params,
    "Odds Ratio": np.exp(params),
    "OR - IC 2.5%": np.exp(conf["IC 2.5%"]),
    "OR - IC 97.5%": np.exp(conf["IC 97.5%"]),
    "p-valor": modelo_sm.pvalues,
})

# marca quais variáveis são estatisticamente significativas (p < 0,05)
odds_ratios["Significativo (p<0,05)"] = odds_ratios["p-valor"] < 0.05

odds_ratios = odds_ratios.drop(index="const").sort_values("p-valor")

print("=== Odds Ratios e significância estatística ===")
print(odds_ratios.round(4).to_string())
print()


2. Árvore de Decisão

In [ ]:
# Métricas

print(classification_report(y_test, y_pred_dt))
print('\n' + '-'*60 + '\n')

# Matriz de Confusão

matriz_conf_dt = confusion_matrix(y_test, y_pred_dt)
plt.figure()
sns.heatmap(matriz_conf_dt, annot=True, fmt='d', cmap=paleta, cbar=False, annot_kws={"size": 14})
plt.title('Matriz de Confusão - Árvore de Decisão', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n' + '-'*60 + '\n')

# Importância das variáveis

importancia_dt = pd.DataFrame({
    'Variável': X_train_scaled.columns,
    'Importância Relativa': dt_tree.feature_importances_
}).sort_values(by='Importância Relativa', ascending=False)

plt.figure()
ax = sns.barplot(data=importancia_dt, x='Importância Relativa', y='Variável', hue='Variável', palette=paleta)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', label_type='edge', fontsize=10, weight='bold', color='black', padding=2)
plt.title('Impacto das Variáveis - Árvore de Decisão', fontsize=14, fontweight='bold')
plt.xlabel('Importância Relativa', fontsize=12, fontweight='bold')
plt.ylabel('Variável', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

3. Random Forest

In [ ]:
# Métricas

print(classification_report(y_test, y_pred_rf))
print('\n' + '-'*60 + '\n')

# Matriz de Confusão

matriz_conf_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure()
sns.heatmap(matriz_conf_rf, annot=True, fmt='d', cmap=paleta, cbar=False, annot_kws={"size": 14})
plt.title('Matriz de Confusão - Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n' + '-'*60 + '\n')

# Importância das variáveis

importancia_rf = pd.DataFrame({
    'Variável': X_train_scaled.columns,
    'Importância Relativa': rf_clf.feature_importances_
}).sort_values(by='Importância Relativa', ascending=False)

plt.figure()
ax = sns.barplot(data=importancia_rf, x='Importância Relativa', y='Variável', hue='Variável', palette=paleta)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', label_type='edge', fontsize=10, weight='bold', color='black', padding=2)
plt.title('Impacto das Variáveis - Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importância Relativa', fontsize=12, fontweight='bold')
plt.ylabel('Variável', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('Métricas importantes para o contexto:')

print(f'Regressão Logística: Acurácia = {acc_log:.2%}, Recall = {rec_log:.2%}, F1-Score = {f1_log:.2%}')
print(f'Árvore de Decisão: Acurácia = {acc_dt:.2%}, Recall = {rec_dt:.2%}, F1-Score = {f1_dt:.2%}')
print(f'Random Forest: Acurácia = {acc_rf:.2%}, Recall = {rec_rf:.2%}, F1-Score = {f1_rf:.2%}')

---

Para avaliar a importância relativa das variáveis no modelo final, em termos de desempenho, a Random Forest apresentou a melhor acurácia geral (0,79) e a maior precisão para a classe de demência (0,86), seguida pela Regressão Logística (acurácia de 0,77) e pela Árvore de Decisão isolada, que teve o pior desempenho nas três métricas (acurácia de 0,71). Entretanto, ao observar o recall da classe de interesse clínico — ou seja, a capacidade de identificar corretamente os pacientes que de fato apresentam demência — a Regressão Logística se destacou com o maior valor (0,72), enquanto a Random Forest (0,67) e principalmente a Árvore de Decisão (0,59) deixaram passar mais casos reais sem diagnóstico (falsos negativos). Como o custo de não identificar um paciente com sinais de demência é clinicamente mais grave do que gerar um alarme falso, esse resultado indica que a Regressão Logística, apesar de não ter a maior acurácia, oferece um equilíbrio mais seguro para esse contexto.

Quanto à importância das variáveis, MMSE e nWBV se mantiveram consistentemente como os dois preditores mais relevantes nos três modelos, reforçando a confiabilidade desse achado, o Escore do Mini Exame do Estado Mental (MMSE) e o volume cerebral normalizado (nWBV) são, de fato, os fatores mais associados ao diagnóstico de demência nesta amostra. Já variáveis como Age, EDUC, SES e M/F apresentaram posições distintas de importância dependendo do modelo — a Regressão Logística indicou essas variáveis como estatisticamente significativas (p < 0,05), enquanto a Árvore de Decisão e a Random Forest atribuíram a elas uma importância bem menor, especialmente ao sexo (M/F), que teve importância praticamente nula nos modelos baseados em árvore.

Essa divergência ocorre porque as medidas de importância utilizadas em árvores e Random Forest tendem a favorecer variáveis contínuas com muitos valores possíveis em detrimento de variáveis binárias, além de distribuírem de forma instável a importância entre variáveis colineares, como eTIV e ASF — que apresentam forte correlação entre si (-0,99), já que ASF é matematicamente derivado do eTIV no próprio dataset OASIS. Por esse motivo, a interpretação da importância das variáveis não deve se basear isoladamente nos gráficos de importância das árvores, sendo mais robusto utilizá-los em conjunto com os coeficientes e p-valores da Regressão Logística, que fornecem tanto a direção do efeito (se a variável aumenta ou reduz o risco) quanto uma medida formal de significância estatística.

---

## Cálculo do gap de Recall entre as classes

In [ ]:
# Comparação do Recall entre as classes

relatorios = {
    'Regressão Logística': classification_report(
        y_test, y_pred_log, output_dict=True
    ),
    'Árvore de Decisão': classification_report(
        y_test, y_pred_dt, output_dict=True
    ),
    'Random Forest': classification_report(
        y_test, y_pred_rf, output_dict=True
    )
}

nomes_classes = {
    '0': 'Nondemented',
    '1': 'Demented/Converted'
}

linhas_recall = []

for nome, report in relatorios.items():
    for classe, nome_classe in nomes_classes.items():
        linhas_recall.append({
            'Modelo': nome,
            'Grupo': nome_classe,
            'Recall': report[classe]['recall']
        })

df_recall = pd.DataFrame(linhas_recall)

df_recall['Recall'] = df_recall['Recall'].round(3)

display(df_recall)

In [ ]:
# Cálculo do gap de Recall entre as classes

gap_recall = (
    df_recall
    .pivot(
        index='Modelo',
        columns='Grupo',
        values='Recall'
    )
)

gap_recall['Gap Recall'] = (
    gap_recall['Nondemented']
    - gap_recall['Demented/Converted']
).abs()

gap_recall = gap_recall.sort_values(
    'Gap Recall',
    ascending=True
)

display(gap_recall)

### Resposta:

**Não.** O desempenho não é equilibrado entre os dois grupos. Nos três modelos, o Recall da classe “com demência” foi inferior ao da classe “sem demência”, indicando maior dificuldade na identificação correta dos pacientes com demência.

A **Regressão Logística apresentou o menor gap de Recall, de 9,5 pontos**, com **Recall de 71,7% para a classe “com demência” e 81,2% para a classe “sem demência”.** O **Random Forest apresentou um gap de 20,0 pontos**, enquanto a **Árvore de Decisão apresentou o maior desequilíbrio, com gap de 24,6 pontos**.

Os resultados indicam que **os modelos apresentam maior dificuldade para identificar corretamente pacientes com demência, aumentando o risco de falsos negativos nessa classe.** A **Regressão Logística apresentou o desempenho mais equilibrado** entre as classes segundo esse critério.

*Observação: A diferença de Recall não parece ser explicada principalmente por um forte desbalanceamento das classes, uma vez que a distribuição da variável alvo é relativamente equilibrada. Portanto, o Recall deve ser considerado em conjunto com as demais métricas na avaliação do desempenho dos modelos, especialmente em um contexto no qual a identificação de pacientes com demência é relevante.*

## Identificação de pacientes em estágio inicial

In [ ]:
# Identificação de pacientes em estágio inicial de demência

cdr_teste = data.loc[X_test_scaled.index, 'CDR']

mask_inicial = cdr_teste == 0.5

print(
    f'Pacientes com CDR = 0.5 no conjunto de teste: '
    f'{mask_inicial.sum()}\n'
)

previsoes = {
    'Regressão Logística': y_pred_log,
    'Árvore de Decisão': y_pred_dt,
    'Random Forest': y_pred_rf
}

for nome, y_pred in previsoes.items():

    y_pred_series = pd.Series(
        y_pred,
        index=X_test_scaled.index
    )

    real_inicial = y_test[mask_inicial]
    pred_inicial = y_pred_series[mask_inicial]

    acertos = (
        real_inicial.values == pred_inicial.values
    ).sum()

    total = mask_inicial.sum()

    percentual = acertos / total * 100

    print(
        f'{nome}: {acertos}/{total} pacientes em estágio inicial '
        f'classificados corretamente ({percentual:.1f}%)'
    )

### Resposta:

Sim, mas com limitações. Entre os 32 pacientes com CDR = 0,5 no conjunto de teste, **a Regressão Logística apresentou o melhor desempenho, classificando corretamente 23 pacientes (71,9%)**. O**Random Forest classificou corretamente 21 pacientes (65,6%)**, enquanto a **Árvore de Decisão apresentou o menor desempenho, classificando corretamente 16 pacientes (50,0%).**

Os resultados indicam que os modelos conseguem identificar parte dos pacientes em estágio inicial de demência, porém apresentam limitações nesse subgrupo. A **Regressão Logística apresentou o melhor resultado, mas ainda classificou incorretamente 28,1% dos pacientes avaliados, demonstrando que a identificação de casos iniciais permanece um desafio para os modelos.**

*Observação: O CDR foi utilizado apenas posteriormente para identificar o subgrupo de pacientes em estágio inicial, não sendo utilizado como variável preditora durante o treinamento. Dessa forma, evita-se o vazamento de dados e é possível avaliar a capacidade dos modelos de classificar esses pacientes com base nas demais características disponíveis.*

## Validação cruzada e generalização

In [ ]:
# Dados utilizados na validação cruzada
X_cv = data.drop(columns=['Group', 'CDR'])
y_cv = data['Group']

modelos_cv = {
    'Regressão Logística': LogisticRegression(
        class_weight='balanced',
        random_state=42,
        max_iter=1000
    ),

    'Árvore de Decisão': DecisionTreeClassifier(
        max_depth=5,
        class_weight='balanced',
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        class_weight='balanced',
        random_state=42
    )
}

In [ ]:
# Função para avaliar a generalização dos modelos por validação cruzada

def diagnostico_cv(modelo, X, y, nome_modelo, cv=5):

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('scaler', StandardScaler()),
        ('modelo', modelo)
    ])

    resultado = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=['accuracy', 'f1'],
        return_train_score=True,
        n_jobs=-1
    )

    acc_treino = resultado['train_accuracy'].mean()
    acc_validacao = resultado['test_accuracy'].mean()

    f1_treino = resultado['train_f1'].mean()
    f1_validacao = resultado['test_f1'].mean()

    gap_accuracy = acc_treino - acc_validacao
    gap_f1 = f1_treino - f1_validacao

    print(f'--- {nome_modelo} ({cv}-fold CV) ---')

    print(
        f'Acurácia treino:    {acc_treino:.3f} '
        f'(+/- {resultado["train_accuracy"].std():.3f})'
    )

    print(
        f'Acurácia validação: {acc_validacao:.3f} '
        f'(+/- {resultado["test_accuracy"].std():.3f})'
    )

    print(
        f'F1-score treino:    {f1_treino:.3f} '
        f'(+/- {resultado["train_f1"].std():.3f})'
    )

    print(
        f'F1-score validação: {f1_validacao:.3f} '
        f'(+/- {resultado["test_f1"].std():.3f})'
    )

    print(f'Gap de acurácia:    {gap_accuracy:.3f}')
    print(f'Gap de F1-score:    {gap_f1:.3f}')

    return {
        'Modelo': nome_modelo,
        'Acurácia treino': acc_treino,
        'Acurácia validação': acc_validacao,
        'Gap acurácia': gap_accuracy,
        'F1-score treino': f1_treino,
        'F1-score validação': f1_validacao,
        'Gap F1-score': gap_f1
    }

In [ ]:
# Avaliação dos modelos por validação cruzada

diagnostico = []

for nome, modelo in modelos_cv.items():

    resultado = diagnostico_cv(
        modelo,
        X_cv,
        y_cv,
        nome
    )

    diagnostico.append(resultado)

df_diagnostico = pd.DataFrame(diagnostico).round(3)

In [ ]:
# Tabela comparativa dos resultados da validação cruzada

df_diagnostico = pd.DataFrame(diagnostico).round(3)

display(df_diagnostico)

### Resposta:

Os resultados indicam **overfitting nos modelos baseados em árvores**, principalmente no Random Forest. A Árvore de Decisão apresentou gap de **16,0 pontos** na acurácia e **15,9 pontos** no F1 score, enquanto o Random Forest apresentou os maiores gaps, de **17,7 pontos** na acurácia e **19,4 pontos** no F1 score. Esses resultados indicam que esses modelos apresentam desempenho consideravelmente superior nos dados de treino em comparação com os dados de validação, sugerindo dificuldade de generalização.

A **Regressão Logística apresentou o comportamento mais estável**, com gap de apenas **2,0 pontos** na acurácia e **3,1 pontos** no F1 score, indicando menor diferença entre o desempenho de treino e validação.

Não foram observados sinais relevantes de **underfitting**, uma vez que os modelos apresentam desempenho adequado no conjunto de treino. Para reduzir o overfitting, podem ser ajustados os hiperparâmetros dos modelos baseados em árvores, reduzindo sua complexidade por meio de parâmetros como `max_depth`, `min_samples_split` e `min_samples_leaf`. No Random Forest, também podem ser ajustados parâmetros como `n_estimators` e `max_features`. Na Regressão Logística, a intensidade da regularização pode ser ajustada por meio do parâmetro `C`.

## CONCLUSÃO GERAL

Os três modelos apresentaram capacidade de classificação dos grupos, porém com diferenças importantes entre as classes e na capacidade de generalização. A Regressão Logística apresentou o comportamento mais estável, com os menores gaps entre treino e validação, de 2,0 pontos percentuais para acurácia e 3,1 pontos percentuais para F1 score. A Árvore de Decisão apresentou comportamento intermediário, enquanto o Random Forest, apesar de apresentar desempenho superior no treinamento, mostrou maior tendência ao overfitting, com os maiores gaps entre treino e validação.

A análise por classe mostrou que os três modelos apresentaram maior dificuldade na identificação de pacientes com demência, evidenciada pelo menor Recall dessa classe. Esse desequilíbrio foi menor na Regressão Logística, que apresentou gap de Recall de 9,5 pontos percentuais. A Árvore de Decisão apresentou o maior gap, de 24,6 pontos percentuais, seguida pelo Random Forest, com 20,0 pontos percentuais.

Na análise específica de pacientes em estágio inicial, representados por CDR = 0,5, os modelos apresentaram desempenho limitado. A Regressão Logística apresentou novamente o melhor resultado, classificando corretamente 23 dos 32 pacientes, correspondendo a 71,9%. O Random Forest classificou corretamente 65,6% e a Árvore de Decisão, 50,0% dos pacientes desse subgrupo.

Dessa forma, para este conjunto de dados, a Regressão Logística apresentou o melhor compromisso entre estabilidade, equilíbrio entre as classes e capacidade de generalização. Entretanto, os resultados também demonstram limitações na identificação de pacientes com demência, especialmente em estágios iniciais. Portanto, os modelos avaliados não devem ser considerados suficientes, de forma isolada, para aplicação clínica, sendo necessários estudos adicionais, validação externa e avaliação clínica antes de qualquer utilização nesse contexto.

**Síntese dos resultados**

- Regressão Logística: maior estabilidade, melhor equilíbrio entre as classes e melhor capacidade de generalização.

- Árvore de Decisão: maior desequilíbrio de Recall entre as classes e evidências de overfitting.

- Random Forest: alto desempenho no treinamento, porém maior tendência ao overfitting e dificuldade de generalização.

- Estágio inicial: desempenho limitado nos três modelos, com melhor resultado da Regressão Logística, que classificou corretamente 71,9% dos pacientes com CDR = 0,5.


**Observações:**

Durante a realização desse desafio surgiu a dúvida sobre analisar as variáveis 'ASF' e 'eTIV' separadamente, por elas, serem matematicamente derivada da outra e apresentarem uma correlação negativa muito forte (-0,99), mas após fazer as seguintes analises:
 - Com as duas;
 - Somente com a 'ASF';
 - Somente com a 'eTIV' e 
 - Sem as duas variáveis.

As diferenças encontradas não tiveram alterações significativas para que houvesse a troca da análise original, com todas as variáveis do df 'data', sendo retirado apenas a variável 'CDR' para evitar vazamento de dados.
